# TennisMyLife — Gallica 1903 Colab worker — A100 MAX
ALTO is strictly rate-limited to one Gallica request every 16 seconds. RapidOCR does not redownload from Gallica: it pulls already-downloaded images from the VPS and runs a 16-process CUDA pool on the A100.


In [ ]:
!rm -rf /content/Tennis-OCR-Pipeline
!git clone -q https://github.com/Tennismylife/Tennis-OCR-Pipeline.git /content/Tennis-OCR-Pipeline
!pip -q uninstall -y onnxruntime onnxruntime-gpu >/dev/null 2>&1 || true
!pip -q install -r /content/Tennis-OCR-Pipeline/colab/requirements.txt


In [ ]:
import subprocess, onnxruntime as ort, os
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv,noheader'],check=True)
providers=ort.get_available_providers(); print('ONNX Runtime providers:',providers)
if 'CUDAExecutionProvider' not in providers: raise RuntimeError('CUDAExecutionProvider not available')
print('GPU CHECK OK — A100 CUDA active; CPU cores:', os.cpu_count())


In [ ]:
from google.colab import files
import base64
uploaded=files.upload()
if not uploaded: raise RuntimeError('No SSH key uploaded')
key_name,key_bytes=next(iter(uploaded.items()))
if key_name.endswith('.pub'): raise RuntimeError('Upload tml_colab_ed25519, not .pub')
KEY_B64=base64.b64encode(key_bytes).decode(); print('SSH key loaded:',key_name)


In [ ]:
import sys
sys.path.insert(0,'/content/Tennis-OCR-Pipeline/colab')
from worker import connect_sftp
VPS_HOST='vibrant-lovelace.82-165-11-122.plesk.page'; VPS_USER='andre'; VPS_PORT=2222
VPS_MANIFEST='/home/andre/GallicaJobs/gallica-1903-all-tennis/GALlica_1903_ALL_TENNIS/00_MANIFEST/colab_active_claims.tsv'
VPS_REMOTE_CACHE='/home/andre/GallicaJobs/gallica-1903-all-tennis/GALlica_1903_ALL_TENNIS/ocr_cache_latin_full/targeted_remaining_1903'
VPS_ALTO_CACHE='/home/andre/GallicaJobs/_shared/alto_cache'
tr,sftp=connect_sftp(VPS_HOST,VPS_USER,KEY_B64,VPS_PORT); sftp.get(VPS_MANIFEST,'/content/colab_claim.tsv'); sftp.close(); tr.close()
rows=sum(1 for _ in open('/content/colab_claim.tsv',encoding='utf-8-sig'))-1; print('Claim downloaded. Rows:',rows)


In [ ]:
import csv
with open('/content/colab_claim.tsv',encoding='utf-8-sig',newline='') as f: rr=list(csv.DictReader(f,delimiter='\t'))
fields=list(rr[0]) if rr else ['ark','page','mode']
for mode,path in [('ALTO','/content/colab_alto.tsv'),('RAPID','/content/colab_rapid.tsv')]:
    subset=[r for r in rr if (r.get('mode') or '').upper()==mode]
    with open(path,'w',encoding='utf-8-sig',newline='') as f:
        w=csv.DictWriter(f,fieldnames=fields,delimiter='\t'); w.writeheader(); w.writerows(subset)
    print(mode,'rows:',len(subset))


In [ ]:
import subprocess, os
subprocess.run(['git','-C','/content/Tennis-OCR-Pipeline','pull','--ff-only'],check=True)
GPU_WORKERS=16
print('A100 MAX mode: GPU_WORKERS=',GPU_WORKERS,'; ALTO interval=16s; OCR Gallica requests=0')
common=['--vps-host',VPS_HOST,'--vps-user',VPS_USER,'--vps-key-b64',KEY_B64,'--vps-port',str(VPS_PORT),'--remote-cache',VPS_REMOTE_CACHE]
alto=['python','/content/Tennis-OCR-Pipeline/colab/alto_safe.py','--manifest','/content/colab_alto.tsv',*common,'--remote-alto-cache',VPS_ALTO_CACHE,'--delay','16']
rapid=['python','/content/Tennis-OCR-Pipeline/colab/rapid_pool.py','--manifest','/content/colab_rapid.tsv',*common,'--profile','HQ','--workers',str(GPU_WORKERS),'--downloaders','6']
print('Starting safe ALTO + max A100 RapidOCR pool...')
p_alto=subprocess.Popen(alto); p_rapid=subprocess.Popen(rapid)
rapid_rc=p_rapid.wait(); alto_rc=p_alto.wait()
print('Branches complete: ALTO=',alto_rc,'RAPID/A100=',rapid_rc)
if alto_rc!=0 or rapid_rc!=0: raise RuntimeError(f'branch failure ALTO={alto_rc} RAPID={rapid_rc}')


The OCR pool transfers source images from the VPS over SFTP, so OCR concurrency never increases Gallica request frequency. Re-running is resumable because already-uploaded outputs are skipped.
